# PhaseNet Picks

In [1]:
import obspy
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from yazel_integration import recovar_pick_cleaner, load_recovar_classifier

2025-11-25 12:06:19.126710: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-25 12:06:19.159899: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-25 12:06:19.159932: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-25 12:06:19.159958: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-25 12:06:19.166754: I tensorflow/core/platform/cpu_feature_g

In [2]:
instance='/mnt/data_a/ege/recovar_models/exp_instance/representation_learning_autoencoder_ensemble/instance/split0/ep19.h5'
merged='/mnt/data_a/ege/recovar_models/MERGED_dilation_v2/representation_learning_autoencoder_ensemble/MERGED_fixed/split0/ep9.h5'

MODEL_PATH= instance

In [3]:
phasenet_pick_dir = "/home/ege/recovar/reproducibility/phasenet_eqt/phasenet_windows_SLVT"
phasenet_picks= pd.read_csv("/home/ege/recovar/reproducibility/phasenet_eqt/phasenet_windows_SLVT/metadata.csv")

In [4]:
p = Path(phasenet_pick_dir)

In [5]:
results = []
classifier = load_recovar_classifier(MODEL_PATH)

for file in p.iterdir():
    if file.is_file() and file.suffix == '.mseed':
        stream = obspy.read(file)
        stream.merge()        
        pick_row = phasenet_picks[phasenet_picks['filename'] == file.name]
        pick_time = obspy.UTCDateTime(pick_row['pick_time'].values[0])
        result = recovar_pick_cleaner(stream=stream, classifier=classifier, pick_idx=pick_time, threshold=None)
        results.append(result)

2025-11-25 12:06:23.044525: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 16953 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:19:00.0, compute capability: 8.6
2025-11-25 12:06:23.045596: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22286 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:1a:00.0, compute capability: 8.6
2025-11-25 12:06:23.046509: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 22286 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:67:00.0, compute capability: 8.6
2025-11-25 12:06:23.047418: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 22211 MB memory:  -> device: 3, name: NVIDIA GeForce RTX 3090

In [6]:
from yazel_integration import preprocess

comparison_data = []
results_idx = 0
catalog_path = '/home/boxx/Public/earthquake_model_evaluations/data/SilivriPaper_2019-09-01__2019-11-30/processed_catalogs/kara74a_phase_picks.csv'
catalog = pd.read_csv(catalog_path)
catalog['p_arrival_time'] = pd.to_datetime(catalog['p_arrival_time'])

for file in sorted(p.iterdir()):
    if not file.is_file() or file.suffix != '.mseed':
        continue

    stream = obspy.read(file)
    stream.merge()
    stream = stream.select(channel="HH*")
    station = stream[0].stats.station

    pick_row = phasenet_picks[phasenet_picks['filename'] == file.name]

    stream_raw = stream.copy()

    stream_processed = stream.copy()
    for tr in stream_processed:
        tr.data = preprocess(tr.data, sampling_rate=tr.stats.sampling_rate, freqmin=1.0, freqmax=20.0)

    fig, (ax_raw, ax_proc) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

    for i, tr in enumerate(stream_raw):
        ax_raw.plot(tr.times("matplotlib"), tr.data, label=tr.stats.channel)
    ax_raw.set_title('Raw')
    ax_raw.legend()

    for i, tr in enumerate(stream_processed):
        ax_proc.plot(tr.times("matplotlib"), tr.data, label=tr.stats.channel)
    ax_proc.set_title('Processed (1-20 Hz bandpass)')
    ax_proc.legend()

    phasenet_pick = pd.to_datetime(pick_row['pick_time'].values[0], format='mixed')
    window_start = pd.to_datetime(pick_row['start_time'].values[0], format='mixed')
    window_end = pd.to_datetime(pick_row['end_time'].values[0], format='mixed')

    catalog_picks = catalog[
        (catalog['station'] == station) &
        (catalog['p_arrival_time'] >= window_start) &
        (catalog['p_arrival_time'] <= window_end)
    ]

    for ax in [ax_raw, ax_proc]:
        ax.axvline(phasenet_pick.to_pydatetime(), color='red', linewidth=1, linestyle='--', label='PhaseNet')
        for _, cat_pick in catalog_picks.iterrows():
            ax.axvline(cat_pick['p_arrival_time'].to_pydatetime(), color='blue', linewidth=1, linestyle=':', label='Catalog')


    fig.suptitle(f'{file.name} - Result: {results[results_idx][0]:.4f}', fontsize=10)
    fig.savefig(f'/home/ege/recovar/reproducibility/phasenet_eqt/phasenet_windows_SLVT_compare/plot_{results_idx:03d}_{file.stem}.png', dpi=150,
bbox_inches='tight')
    plt.close(fig)

    results_idx += 1

In [7]:
comparison_data = []
results_idx = 0
catalog_path = '/home/boxx/Public/earthquake_model_evaluations/data/SilivriPaper_2019-09-01__2019-11-30/processed_catalogs/kara74a_phase_picks.csv'
catalog = pd.read_csv(catalog_path)
catalog['p_arrival_time'] = pd.to_datetime(catalog['p_arrival_time'])
for file in sorted(p.iterdir()):
    if not file.is_file() or file.suffix != '.mseed':
        continue
        
    stream = obspy.read(file)
    stream.merge()
    stream = stream.select(channel="HH*")
    station = stream[0].stats.station
    
    pick_row = phasenet_picks[phasenet_picks['filename'] == file.name]
    
    if pick_row.empty:
        continue
    
    fig = stream.plot(handle=True)
    
    phasenet_pick = pd.to_datetime(pick_row['pick_time'].values[0], format='mixed')
    window_start = pd.to_datetime(pick_row['start_time'].values[0], format='mixed')
    window_end = pd.to_datetime(pick_row['end_time'].values[0], format='mixed')
    
    for ax in fig.axes:
        ax.axvline(phasenet_pick.to_pydatetime(), color='red', linewidth=1, linestyle='--')
    
    catalog_picks = catalog[
        (catalog['station'] == station) &
        (catalog['p_arrival_time'] >= window_start) &
        (catalog['p_arrival_time'] <= window_end)
    ]
    
    catalog_pick = catalog_picks.iloc[0]['p_arrival_time'] if not catalog_picks.empty else None
    
    for _, cat_pick in catalog_picks.iterrows():
        for ax in fig.axes:
            ax.axvline(cat_pick['p_arrival_time'].to_pydatetime(), color='blue', linewidth=1, linestyle=':', label='Catalog')
    
    has_catalog = catalog_pick is not None
    has_phasenet = phasenet_pick is not None
    
    if has_catalog and has_phasenet:
        status = "Both"
    elif has_catalog:
        status = "Catalog only"
    elif has_phasenet:
        status = "PhaseNet only"
    else:
        status = "None"
    
    comparison_data.append({
        'filename': file.name,
        'station': station,
        'catalog_pick': catalog_pick,
        'phasenet_pick': phasenet_pick,
        'model_score': results[results_idx][0],
        'detection_status': status
    })
    
    fig.suptitle(f'{file.name} - Result: {results[results_idx][0]:.4f}', fontsize=10)
    #fig.savefig(f'/home/ege/recovar/reproducibility/phasenet_eqt/phasenet_windows_SLVT/plot_{results_idx:03d}_{file.stem}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    
    results_idx += 1

comparison_df = pd.DataFrame(comparison_data)
comparison_df.to_csv('SLVT_pick_comparison.csv', index=False)

In [8]:
pick_comparison = pd.read_csv('SLVT_pick_comparison.csv')
pick_comparison[pick_comparison['detection_status']=='PhaseNet only']

,filename,station,catalog_pick,phasenet_pick,model_score,detection_status
0,window_20190912_0000.mseed,SLVT,NaN,2019-09-12 06:17:33.700,0.133620,PhaseNet only
1,window_20190915_0000.mseed,SLVT,NaN,2019-09-15 03:42:28.660,0.799271,PhaseNet only
2,window_20190916_0000.mseed,SLVT,NaN,2019-09-16 09:47:57.570,0.062698,PhaseNet only
3,window_20190916_0001.mseed,SLVT,NaN,2019-09-17 01:37:38.340,0.072717,PhaseNet only
4,window_20190916_0002.mseed,SLVT,NaN,2019-09-17 01:38:19.040,0.225550,PhaseNet only
...,...,...,...,...,...,...
1044,window_20191128_0001.mseed,SLVT,NaN,2019-11-29 06:24:14.660,0.206431,PhaseNet only
1045,window_20191128_0002.mseed,SLVT,NaN,2019-11-29 19:52:13.960,0.263479,PhaseNet only
1046,window_20191129_0000.mseed,SLVT,NaN,2019-11-30 00:14:41.930,0.041860,PhaseNet only
1047,window_20191129_0001.mseed,SLVT,NaN,2019-11-30 00:37:57.740,0.190940,PhaseNet only


In [9]:
pick_comparison[pick_comparison['detection_status']=='Both']

,filename,station,catalog_pick,phasenet_pick,model_score,detection_status
9,window_20190920_0000.mseed,SLVT,2019-09-20 16:53:47.899929,2019-09-20 16:53:48.000,0.175456,Both
11,window_20190921_0000.mseed,SLVT,2019-09-22 03:09:44.511868,2019-09-22 03:09:44.650,0.039776,Both
12,window_20190921_0001.mseed,SLVT,2019-09-22 08:39:04.523460,2019-09-22 08:39:04.660,0.110910,Both
13,window_20190921_0002.mseed,SLVT,2019-09-22 08:42:50.163275,2019-09-22 08:42:50.340,0.338448,Both
14,window_20190921_0003.mseed,SLVT,2019-09-22 12:07:17.475660,2019-09-22 12:07:17.590,0.212853,Both
...,...,...,...,...,...,...
1002,window_20191120_0000.mseed,SLVT,2019-11-21 02:51:00.718115,2019-11-21 02:51:00.880,0.433365,Both
1017,window_20191123_0001.mseed,SLVT,2019-11-24 00:54:48.896859,2019-11-24 00:54:49.050,0.099766,Both
1028,window_20191124_0002.mseed,SLVT,2019-11-25 12:12:01.868195,2019-11-25 12:12:02.120,0.107952,Both
1038,window_20191127_0001.mseed,SLVT,2019-11-28 02:57:59.862414,2019-11-28 02:57:59.900,0.277958,Both


In [10]:
df = pd.read_csv('SLVT_pick_comparison.csv')
df

,filename,station,catalog_pick,phasenet_pick,model_score,detection_status
0,window_20190912_0000.mseed,SLVT,NaN,2019-09-12 06:17:33.700,0.133620,PhaseNet only
1,window_20190915_0000.mseed,SLVT,NaN,2019-09-15 03:42:28.660,0.799271,PhaseNet only
2,window_20190916_0000.mseed,SLVT,NaN,2019-09-16 09:47:57.570,0.062698,PhaseNet only
3,window_20190916_0001.mseed,SLVT,NaN,2019-09-17 01:37:38.340,0.072717,PhaseNet only
4,window_20190916_0002.mseed,SLVT,NaN,2019-09-17 01:38:19.040,0.225550,PhaseNet only
...,...,...,...,...,...,...
1045,window_20191128_0002.mseed,SLVT,NaN,2019-11-29 19:52:13.960,0.263479,PhaseNet only
1046,window_20191129_0000.mseed,SLVT,NaN,2019-11-30 00:14:41.930,0.041860,PhaseNet only
1047,window_20191129_0001.mseed,SLVT,NaN,2019-11-30 00:37:57.740,0.190940,PhaseNet only
1048,window_20191129_0002.mseed,SLVT,2019-11-30 06:37:23.402686,2019-11-30 06:37:23.510,0.087482,Both


In [11]:
from sklearn.metrics import f1_score
def find_best_f1_threshold(scores, labels, num_thresholds=500):
    """
    Find the score threshold that maximizes F1 score.

    scores: np.ndarray of shape (N,)
    labels: np.ndarray of shape (N,), binary {0,1}
    num_thresholds: number of thresholds to scan between min and max
    
    returns: best_threshold, best_f1
    """

    scores = np.asarray(scores).reshape(-1)
    labels = np.asarray(labels).reshape(-1)

    # Create candidate thresholds — include boundaries
    thresholds = np.linspace(scores.min(), scores.max(), num_thresholds)

    best_f1 = -1.0
    best_thr = thresholds[0]

    for t in thresholds:
        preds = (scores >= t).astype(int)
        f1 = f1_score(labels, preds)   # macro/micro not needed for binary

        if f1 > best_f1:
            best_f1 = f1
            best_thr = t

    return best_thr, best_f1

df = pd.read_csv('SLVT_pick_comparison.csv')

find_best_f1_threshold(df["model_score"],~df["catalog_pick"].isna())

(0.004245341, 0.6037234042553191)

In [12]:
df = pd.read_csv('SLVT_pick_comparison.csv')

both = df[df['detection_status'] == 'Both']['model_score'].values
phasenet_only = df[df['detection_status'] == 'PhaseNet only']['model_score'].values

print("=== DETECTION STATISTICS ===\n")

print(f"Total windows: {len(df)}")
print(f"Both (catalog + PhaseNet): {len(both)}")
print(f"PhaseNet only: {len(phasenet_only)}")
print(f"Catalog only: {len(df[df['detection_status'] == 'Catalog only'])}\n")

print("=== MODEL SCORES: BOTH (TRUE POSITIVES) ===")
print(f"Count: {len(both)}")
print(f"Mean: {np.mean(both):.3f}")
print(f"Std: {np.std(both):.3f}")
print(f"Min: {np.min(both):.3f}")
print(f"Max: {np.max(both):.3f}")

print("=== MODEL SCORES: PHASENET ONLY (FALSE POSITIVES) ===")
print(f"Count: {len(phasenet_only)}")
print(f"Mean: {np.mean(phasenet_only):.3f}")
print(f"Std: {np.std(phasenet_only):.3f}")
print(f"Min: {np.min(phasenet_only):.3f}")
print(f"Max: {np.max(phasenet_only):.3f}")

print("=== THRESHOLD ANALYSIS ===")
for thresh in [0.089,0.0042470098]:
    tp = np.sum(both >= thresh)
    fp = np.sum(phasenet_only >= thresh)
    fn = np.sum(both < thresh)
    tn = np.sum(phasenet_only < thresh)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f"Threshold {thresh:.2f}:")
    print(f"  TP={tp}, FP={fp}, FN={fn}, TN={tn}")
    print(f"  Precision={precision:.3f}, Recall={recall:.3f}\n")

=== DETECTION STATISTICS ===

Total windows: 1050
Both (catalog + PhaseNet): 454
PhaseNet only: 596
Catalog only: 0

=== MODEL SCORES: BOTH (TRUE POSITIVES) ===
Count: 454
Mean: 0.215
Std: 0.175
Min: 0.004
Max: 0.945
=== MODEL SCORES: PHASENET ONLY (FALSE POSITIVES) ===
Count: 596
Mean: 0.219
Std: 0.170
Min: 0.007
Max: 0.938
=== THRESHOLD ANALYSIS ===
Threshold 0.09:
  TP=326, FP=454, FN=128, TN=142
  Precision=0.418, Recall=0.718

Threshold 0.00:
  TP=453, FP=596, FN=1, TN=0
  Precision=0.432, Recall=0.998



## Overlapping Windows

In [13]:
def sliding_window_analysis(stream, classifier, large_window_size=45, classifier_window_size=3000,
                            stride=1, sampling_rate=100.0, channel_pattern="HH"):

    large_window_samples = int(large_window_size * sampling_rate)
    stride_samples = int(stride * sampling_rate)

    for tr in stream:
        tr.data = tr.data.astype(np.float32)

    z_trace = stream.select(channel=f"{channel_pattern}Z")[0]
    n_trace = stream.select(channel=f"{channel_pattern}N")[0]
    e_trace = stream.select(channel=f"{channel_pattern}E")[0]

    min_length = min(len(z_trace.data), len(n_trace.data), len(e_trace.data))
    
    if min_length >= large_window_samples:
        center_idx = min_length // 2
        start_idx = center_idx - large_window_samples // 2
        end_idx = start_idx + large_window_samples
    else:
        start_idx = 0
        end_idx = large_window_samples

    z_large = z_trace.data[start_idx:end_idx]
    n_large = n_trace.data[start_idx:end_idx]
    e_large = e_trace.data[start_idx:end_idx]

    num_positions = (large_window_samples - classifier_window_size) // stride_samples + 1

    scores = []

    for i in range(num_positions):
        window_start = i * stride_samples
        window_end = window_start + classifier_window_size

        z_window = z_large[window_start:window_end]
        n_window = n_large[window_start:window_end]
        e_window = e_large[window_start:window_end]

        e_processed = preprocess(e_window, sampling_rate=sampling_rate)
        n_processed = preprocess(n_window, sampling_rate=sampling_rate)
        z_processed = preprocess(z_window, sampling_rate=sampling_rate)

        waveform = np.stack([e_processed, n_processed, z_processed], axis=-1)
        waveform = np.expand_dims(waveform, axis=0)

        score = classifier(waveform)
        scores.append(float(score[0]))

    scores = np.array(scores)

    return {
        'scores': scores,
        'max': float(np.max(scores)),
        'mean': float(np.mean(scores)),
        'num_windows': len(scores)
    }

In [14]:
def compare_sliding_window_statistics(csv_path):
    import pandas as pd

    df = pd.read_csv(csv_path)

    for score_type in ['max', 'mean']:
        both = df[df['detection_status'] == 'Both'][f'model_{score_type}'].values
        phasenet_only = df[df['detection_status'] == 'PhaseNet only'][f'model_{score_type}'].values

        print(f"\n{'='*60}")
        print(f"ANALYSIS USING {score_type.upper()} SCORES")
        print(f"{'='*60}\n")

        print("=== DETECTION STATISTICS ===")
        print(f"Total windows: {len(df)}")
        print(f"Both (catalog + PhaseNet): {len(both)}")
        print(f"PhaseNet only: {len(phasenet_only)}")
        print(f"Catalog only: {len(df[df['detection_status'] == 'Catalog only'])}\n")

        print(f"=== {score_type.upper()} SCORES: BOTH (TRUE POSITIVES) ===")
        print(f"Mean: {np.mean(both):.3f}")
        print(f"Std: {np.std(both):.3f}")
        print(f"Min: {np.min(both):.3f}")
        print(f"Max: {np.max(both):.3f}\n")

        print(f"=== {score_type.upper()} SCORES: PHASENET ONLY (FALSE POSITIVES) ===")
        print(f"Mean: {np.mean(phasenet_only):.3f}")
        print(f"Std: {np.std(phasenet_only):.3f}")
        print(f"Min: {np.min(phasenet_only):.3f}")
        print(f"Max: {np.max(phasenet_only):.3f}\n")

        print("=== THRESHOLD ANALYSIS ===")
        for thresh in [0.10, 0.2, 0.25, 0.3, 0.35]:
            tp = np.sum(both >= thresh)
            fp = np.sum(phasenet_only >= thresh)
            fn = np.sum(both < thresh)
            tn = np.sum(phasenet_only < thresh)

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            print(f"Threshold {thresh:.2f}: TP={tp}, FP={fp}, FN={fn}, TN={tn}, "
                  f"Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")

In [15]:
from pathlib import Path
import obspy
import pandas as pd
from yazel_integration import load_recovar_classifier, sliding_window_analysis, compare_sliding_window_statistics

instance='/mnt/data_a/ege/recovar_models/exp_instance/representation_learning_autoencoder_ensemble/instance/split0/ep19.h5'
MODEL_PATH=instance
classifier = load_recovar_classifier(MODEL_PATH, WINDOW_SIZE=3000)

phasenet_pick_dir = "/home/ege/recovar/reproducibility/phasenet_eqt/phasenet_windows_SLVT"
phasenet_picks = pd.read_csv(f"{phasenet_pick_dir}/metadata.csv")
catalog_path =
'/home/boxx/Public/earthquake_model_evaluations/data/SilivriPaper_2019-09-01__2019-11-30/processed_catalogs/kara74a_phase_picks.csv'
catalog = pd.read_csv(catalog_path)
catalog['p_arrival_time'] = pd.to_datetime(catalog['p_arrival_time'])

comparison_data = []
p = Path(phasenet_pick_dir)

for file in sorted(p.iterdir()):
    if not file.is_file() or file.suffix != '.mseed':
        continue

    stream = obspy.read(file)
    stream.merge()
    stream = stream.select(channel="HH*")
    station = stream[0].stats.station

    pick_row = phasenet_picks[phasenet_picks['filename'] == file.name]
    if pick_row.empty:
        continue

    result = sliding_window_analysis(stream, classifier, large_window_size=90, stride=1, channel_pattern="HH")

    phasenet_pick = pd.to_datetime(pick_row['pick_time'].values[0], format='mixed')
    window_start = pd.to_datetime(pick_row['start_time'].values[0], format='mixed')
    window_end = pd.to_datetime(pick_row['end_time'].values[0], format='mixed')

    catalog_picks = catalog[
        (catalog['station'] == station) &
        (catalog['p_arrival_time'] >= window_start) &
        (catalog['p_arrival_time'] <= window_end)
    ]
    catalog_pick = catalog_picks.iloc[0]['p_arrival_time'] if not catalog_picks.empty else None


    if catalog_pick is not None and phasenet_pick is not None:
        status = "Both"
    elif catalog_pick is not None:
        status = "Catalog only"
    elif phasenet_pick is not None:
        status = "PhaseNet only"
    else:
        status = "None"

    comparison_data.append({
        'filename': file.name,
        'station': station,
        'catalog_pick': catalog_pick,
        'phasenet_pick': phasenet_pick,
        'model_max': result['max'],
        'model_mean': result['mean'],
        'detection_status': status
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df.to_csv('SLVT_sliding_window_comparison.csv', index=False)

compare_sliding_window_statistics('SLVT_sliding_window_comparison.csv')



SyntaxError: invalid syntax (701673131.py, line 12)